# Phase 1b — Feature Engineering (Baseline + Expanded)

Builds two feature tables per (customer, window) and persists them to parquet so downstream phases never recompute:

| Set | Source | Approx columns |
|---|---|---|
| **baseline** | `src.features.build_features.build_customer_features` | ~30 |
| **expanded** | `src.features.expanded.build_expanded_features` | ~70 |

Target-encoded columns are NOT added here (they require labels and a train/val/test split discipline) — those are computed in `02b_baseline_vs_expanded.ipynb`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data.splits import build_windows
from src.features.build_features import build_customer_features
from src.features.expanded import build_expanded_features

PROC = PROJECT_ROOT / 'data' / 'processed'
FEAT = PROJECT_ROOT / 'data' / 'features'
FEAT.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / 'transactions_clean.parquet')
windows = build_windows(df, horizon_days=90, n_windows=3)
top_countries = df['country'].value_counts().head(8).index.tolist()
[(w.name, w.feature_end.date()) for w in windows]

## Baseline features

In [ ]:
for w in windows:
    feats = build_customer_features(df, w.feature_end, country_dummies=top_countries)
    out = FEAT / f'baseline_features_{w.name}.parquet'
    feats.to_parquet(out, index=False)
    print(f'{w.name:>5}: {len(feats):>6,} rows  |  {feats.shape[1]} cols  ->  {out.name}')

## Expanded features (10 Kaggle-style families)

In [ ]:
for w in windows:
    feats = build_expanded_features(df, w.feature_end, country_dummies=top_countries)
    out = FEAT / f'expanded_features_{w.name}.parquet'
    feats.to_parquet(out, index=False)
    print(f'{w.name:>5}: {len(feats):>6,} rows  |  {feats.shape[1]} cols  ->  {out.name}')

In [ ]:
# Quick sanity peek
ex = pd.read_parquet(FEAT / 'expanded_features_train.parquet')
ex.describe().T[['count','mean','std','min','50%','max']].head(20)